In [ ]:
!pip install -q coqui-tts
!pip install pesq pystoi
!pip install git+https://github.com/speechbrain/speechbrain.git

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pesq: filename=pesq-0.0.4-cp312-cp312-linux_x86_64.whl size=284121 sha256=3ade21369cc22d5c5748a7eb811281ba3b59bd0e6501045c26c382122835a26b
  Stored in directory: /root/.cache/pip/wheels/9b/d4/a4/9cf3512534cd47ce4a036d1593ee4013f2bf7509e631a147a3
Successfully built pesq
  Cloning https://github.com/speechbrain/speechbrain.git to /tmp/pip-

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import torch.optim as optim
import math
import time
import gc
import numpy as np
import pandas as pd
import librosa
from datasets import load_dataset
from speechbrain.inference.speaker import SpeakerRecognition
from pesq import pesq
from pystoi import stoi
import warnings
warnings.filterwarnings("ignore")

os.environ["COQUI_TOS_AGREED"] = "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000

# ============================================================
# 📥 PART 1: THE AUDIO ACQUISITION PIPELINE
# ============================================================
INPUT_DIR = "/content/audio_samples"
OUTPUT_DIR = "/content/experiments_batch_top5"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("█" * 70)
print("█" + " PART 1: DOWNLOADING DATASET ".center(68) + "█")
print("█" * 70)

AUDIO_FILES = [os.path.join(INPUT_DIR, f"sample_{i}.wav") for i in range(1, 11)]
files_missing = any(not os.path.exists(f) for f in AUDIO_FILES)

if files_missing:
    print("Accessing LibriSpeech (Streaming Mode)...")
    dataset = load_dataset("librispeech_asr", "clean", split="train.100", streaming=True).shuffle(seed=42)

    print("Downloading 10 clean human speech samples...")
    for i, example in enumerate(dataset.take(10)):
        audio_tensor = torch.tensor(example["audio"]["array"]).unsqueeze(0)
        sampling_rate = example["audio"]["sampling_rate"]

        file_path = os.path.join(INPUT_DIR, f"sample_{i+1}.wav")
        torchaudio.save(file_path, audio_tensor, sampling_rate)
        print(f"   ✅ Saved: {file_path}")
else:
    print("✅ All 10 LibriSpeech samples already exist. Skipping download.")

print("\n")


# ============================================================
# 🛡️ PART 2: THE ADVERSARIAL BATCH PIPELINE
# ============================================================

class QualityConstrainedAttacker(nn.Module):
    def __init__(self, order=16, n_tilt_stages=4, n_sq_basis=8, n_bands=8,
                 beta_limit=0.35, sq_limit=0.50, lpc_limit=0.02, gain_limit=0.20):
        super().__init__()
        self.order = order
        self.n_tilt_stages = n_tilt_stages
        self.n_sq_basis = n_sq_basis
        self.n_bands = n_bands
        self.beta_limit = beta_limit
        self.sq_limit = sq_limit
        self.lpc_limit = lpc_limit
        self.gain_limit = gain_limit

        self.betas = nn.Parameter(torch.zeros(n_tilt_stages))
        self.sq_coeffs = nn.Parameter(torch.zeros(n_sq_basis))
        self.lpc_offsets = nn.Parameter(torch.zeros(order))
        self.band_gains = nn.Parameter(torch.zeros(n_bands))

        self.register_buffer(
            'band_edges',
            torch.tensor([0, 300, 700, 1200, 2000, 3000, 4500, 6000, 8000], dtype=torch.float32)
        )

    def forward(self, x):
        batch_size, frame_len = x.shape
        device = x.device

        n_fft = 2 ** math.ceil(math.log2(frame_len * 2 - 1))
        X_freq = torch.fft.rfft(x, n=n_fft)
        power_spec = X_freq.abs() ** 2
        autocorr = torch.fft.irfft(power_spec, n=n_fft)

        r = autocorr[:, :self.order + 1]
        R = torch.zeros(batch_size, self.order, self.order, device=device)
        for i in range(self.order):
            for j in range(self.order):
                R[:, i, j] = r[:, abs(i - j)]
        p = r[:, 1:]

        a = torch.linalg.solve(R, p)
        a_coeffs_orig = torch.cat([torch.ones(batch_size, 1, device=device), -a], dim=1)

        a_coeffs_perturbed = a_coeffs_orig.clone()
        a_coeffs_perturbed[:, 1:] = a_coeffs_orig[:, 1:] + self.lpc_offsets.unsqueeze(0)

        x_reshaped = x.unsqueeze(1)
        a_flipped = torch.flip(a_coeffs_orig, dims=[1]).unsqueeze(1)
        residual = F.conv1d(x_reshaped, a_flipped, padding=self.order)
        source_E = residual[:, :, :-self.order].squeeze(1)

        E_freq = torch.fft.rfft(source_E, n=frame_len)
        magnitude = E_freq.abs()
        phase = E_freq.angle()

        num_bins = phase.shape[-1]
        freq_bins = torch.linspace(0, math.pi, num_bins, device=device)

        dispersion = torch.zeros(num_bins, device=device)
        for k in range(self.n_sq_basis):
            dispersion = dispersion + (self.sq_coeffs[k] * torch.sin((k + 1) * freq_bins))

        warped_phase = phase + dispersion
        E_freq_warped = magnitude * torch.polar(torch.ones_like(magnitude), warped_phase)

        a_padded = F.pad(a_coeffs_perturbed, (0, frame_len - a_coeffs_perturbed.shape[1]))
        A_freq = torch.fft.rfft(a_padded, n=frame_len) + 1e-8
        X_reconstructed_freq = E_freq_warped / A_freq

        n_freq = X_reconstructed_freq.shape[-1]
        freq_hz = torch.linspace(0, SR / 2, n_freq, device=device)
        gain_profile = torch.zeros(n_freq, device=device)
        for b in range(self.n_bands):
            lo, hi = self.band_edges[b], self.band_edges[b + 1]
            mask = (torch.sigmoid((freq_hz - lo) / 50.0) * torch.sigmoid((hi - freq_hz) / 50.0))
            gain_profile = gain_profile + self.band_gains[b] * mask

        X_reconstructed_freq = X_reconstructed_freq * torch.exp(gain_profile.unsqueeze(0))
        reconstructed_audio = torch.fft.irfft(X_reconstructed_freq, n=frame_len)

        output = reconstructed_audio
        for k in range(self.n_tilt_stages):
            tilt_filter = torch.stack([torch.ones(1, device=device), -self.betas[k].unsqueeze(0)]).view(1, 1, 2)
            output_padded = F.pad(output.unsqueeze(1), (1, 0))
            output = F.conv1d(output_padded, tilt_filter).squeeze(1)

        return output

    def clamp_params(self):
        with torch.no_grad():
            self.betas.clamp_(-self.beta_limit, self.beta_limit)
            self.sq_coeffs.clamp_(-self.sq_limit, self.sq_limit)
            self.lpc_offsets.clamp_(-self.lpc_limit, self.lpc_limit)
            self.band_gains.clamp_(-self.gain_limit, self.gain_limit)

# Loss Functions
def mcd_attack_loss(original, perturbed, mfcc_transform):
    mfcc_orig = mfcc_transform(original)
    mfcc_pert = mfcc_transform(perturbed)
    diff = mfcc_orig[:, 1:14, :] - mfcc_pert[:, 1:14, :]
    mcd_per_frame = torch.sqrt(2.0 * torch.sum(diff ** 2, dim=1) + 1e-8)
    return -torch.mean(mcd_per_frame)

def delta_mfcc_attack_loss(original, perturbed, mfcc_transform):
    mo = mfcc_transform(original)[:, 1:14, :]
    mp = mfcc_transform(perturbed)[:, 1:14, :]
    do_ = mo[:, :, 2:] - mo[:, :, :-2]
    dp_ = mp[:, :, 2:] - mp[:, :, :-2]
    ddo = do_[:, :, 2:] - do_[:, :, :-2]
    ddp = dp_[:, :, 2:] - dp_[:, :, :-2]
    return -torch.mean((do_ - dp_) ** 2) - 0.5 * torch.mean((ddo - ddp) ** 2)

def cosine_embed_loss(original, perturbed, verifier, emb_orig=None):
    if emb_orig is None:
        with torch.no_grad():
            emb_orig = verifier.encode_batch(original)
    emb_pert = verifier.encode_batch(perturbed)
    sim = F.cosine_similarity(emb_orig.squeeze(), emb_pert.squeeze(), dim=0)
    return sim, emb_orig

def multi_resolution_mel_loss(original, perturbed, mel_transforms):
    total = 0.0
    for mel_t in mel_transforms:
        lo = torch.log(mel_t(original) + 1e-7)
        lp = torch.log(mel_t(perturbed) + 1e-7)
        total += F.l1_loss(lp, lo)
    return total / len(mel_transforms)

def waveform_snr_loss(original, perturbed, target_snr_db=20.0):
    noise = perturbed - original
    signal_power = torch.mean(original ** 2)
    noise_power = torch.mean(noise ** 2) + 1e-10
    current_snr = 10 * torch.log10(signal_power / noise_power)
    return F.relu(target_snr_db - current_snr)

def waveform_mse_loss(original, perturbed):
    return torch.mean((original - perturbed) ** 2)

# EER Calculation Function
def compute_eer(pos_scores, neg_scores):
    """Calculates Equal Error Rate using pure Numpy"""
    if not pos_scores or not neg_scores: return 0.0
    scores = np.array(pos_scores + neg_scores)
    labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))

    desc_score_indices = np.argsort(scores)[::-1]
    scores = scores[desc_score_indices]
    labels = labels[desc_score_indices]

    fps = np.cumsum(labels == 0)
    tps = np.cumsum(labels == 1)

    far = fps / float(len(neg_scores))
    fnr = 1.0 - (tps / float(len(pos_scores)))

    diff = np.abs(far - fnr)
    eer_idx = np.argmin(diff)
    eer = (far[eer_idx] + fnr[eer_idx]) / 2.0
    return round(eer * 100, 2)


# Top 5 Experiments
EXPERIMENTS = {
    "A3_moderate": {
        "iterations": 800, "beta_limit": 0.35, "sq_limit": 0.50, "lpc_limit": 0.025, "gain_limit": 0.20,
        "mcd": 1.5, "delta_mfcc": 1.0,
        "mel_preserve": 10.0, "waveform_mse": 200.0, "snr_floor": 40.0, "snr_target": 15.0, "param_reg": 1.0,
    },
    "C3_light_preserve": {
        "iterations": 1000, "beta_limit": 0.35, "sq_limit": 0.50, "lpc_limit": 0.02, "gain_limit": 0.20,
        "mcd": 2.0, "delta_mfcc": 1.5,
        "mel_preserve": 8.0, "waveform_mse": 150.0, "snr_floor": 30.0, "snr_target": 15.0, "param_reg": 0.5,
    },
    "B3_mcd_cosine_safe": {
        "iterations": 1000, "beta_limit": 0.35, "sq_limit": 0.50, "lpc_limit": 0.02, "gain_limit": 0.20,
        "mcd": 1.5, "cosine_embed": 2.0,
        "mel_preserve": 15.0, "waveform_mse": 300.0, "snr_floor": 50.0, "snr_target": 20.0, "param_reg": 1.0,
    },
    "A2_tight": {
        "iterations": 800, "beta_limit": 0.25, "sq_limit": 0.35, "lpc_limit": 0.015, "gain_limit": 0.12,
        "mcd": 1.5, "delta_mfcc": 1.0,
        "mel_preserve": 15.0, "waveform_mse": 300.0, "snr_floor": 50.0, "snr_target": 20.0, "param_reg": 1.5,
    },
    "A1_ultraTight": {
        "iterations": 800, "beta_limit": 0.15, "sq_limit": 0.20, "lpc_limit": 0.008, "gain_limit": 0.08,
        "mcd": 1.5, "delta_mfcc": 1.0,
        "mel_preserve": 20.0, "waveform_mse": 500.0, "snr_floor": 50.0, "snr_target": 25.0, "param_reg": 2.0,
    }
}

# Evaluation Helpers
def safe_float(val, default=0.0):
    try: return float(val)
    except (ValueError, TypeError): return default

def calculate_protection_metrics(orig_path, prot_path, verifier):
    wav_o, _ = librosa.load(orig_path, sr=16000)
    wav_p, _ = librosa.load(prot_path, sr=16000)
    ml = min(len(wav_o), len(wav_p))
    wav_o, wav_p = wav_o[:ml], wav_p[:ml]

    noise = wav_o - wav_p
    snr = 10 * np.log10(np.sum(wav_o ** 2) / (np.sum(noise ** 2) + 1e-8))

    try: pesq_val = pesq(16000, np.clip(wav_o, -1, 1), np.clip(wav_p, -1, 1), 'wb')
    except Exception: pesq_val = 1.0

    try: stoi_val = stoi(wav_o, wav_p, 16000, extended=False)
    except Exception: stoi_val = 0.0

    mfcc_o = librosa.feature.mfcc(y=wav_o, sr=16000, n_mfcc=14)[1:]
    mfcc_p = librosa.feature.mfcc(y=wav_p, sr=16000, n_mfcc=14)[1:]
    mt = min(mfcc_o.shape[1], mfcc_p.shape[1])
    mcd_val = float(np.mean(np.sqrt(2.0 * np.sum((mfcc_o[:, :mt] - mfcc_p[:, :mt]) ** 2, axis=0))))

    t_o = torch.tensor(wav_o).unsqueeze(0).to(DEVICE)
    t_p = torch.tensor(wav_p).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        eo = verifier.encode_batch(t_o)
        ep = verifier.encode_batch(t_p)
        cos_sim = F.cosine_similarity(eo.squeeze(), ep.squeeze(), dim=0).item()

    return {"CosSim_Prot": round(cos_sim, 4), "MCD_Prot": round(mcd_val, 2),
            "PESQ": round(float(pesq_val), 2), "STOI": round(float(stoi_val), 4), "SNR_dB": round(float(snr), 1)}

def calculate_clone_metrics(orig_path, clone_path, verifier):
    wav_o, _ = librosa.load(orig_path, sr=16000)
    wav_c, _ = librosa.load(clone_path, sr=16000)

    t_o = torch.tensor(wav_o).unsqueeze(0).to(DEVICE)
    t_c = torch.tensor(wav_c).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        eo = verifier.encode_batch(t_o)
        ec = verifier.encode_batch(t_c)
        cos_sim = F.cosine_similarity(eo.squeeze(), ec.squeeze(), dim=0).item()

    mfcc_o = librosa.feature.mfcc(y=wav_o, sr=16000, n_mfcc=14)[1:]
    mfcc_c = librosa.feature.mfcc(y=wav_c, sr=16000, n_mfcc=14)[1:]
    clone_mcd = float(np.sqrt(2.0 * np.sum((np.mean(mfcc_o, axis=1) - np.mean(mfcc_c, axis=1)) ** 2)))

    return {"Clone_CosSim": round(cos_sim, 4), "Clone_MCD": round(clone_mcd, 2)}


def run_batch_pipeline():
    total_start = time.time()

    print("█" * 70)
    print("█" + " PART 2: BATCH PROCESSING TOP 5 EXPERIMENTS ".center(68) + "█")
    print("█" * 70)

    print("\n📦 Loading speaker verifier (ECAPA-TDNN)...")
    verifier = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="pretrained_models/spkrec-ecapa-voxceleb",
        run_opts={"device": DEVICE}
    )

    processed_files = []

    # ==========================================
    # STAGE A: TRAIN ALL FILES
    # ==========================================
    print("\n" + "=" * 70)
    print("🔧 STAGE A: Training Models Across All Audios...")
    print("=" * 70)

    for audio_path in AUDIO_FILES:
        if not os.path.exists(audio_path):
            print(f"⚠️ Skipping {audio_path} (File not found)")
            continue

        base_name = os.path.basename(audio_path).replace('.wav', '')
        processed_files.append({"path": audio_path, "name": base_name})
        print(f"\n🎧 Processing Voice: {base_name}")

        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000: waveform = T.Resample(sr, 16000)(waveform)
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = waveform.to(DEVICE)

        mfcc_transform = T.MFCC(sample_rate=16000, n_mfcc=40).to(DEVICE)
        mel_transforms = [T.MelSpectrogram(sample_rate=16000, n_fft=n, hop_length=h, n_mels=80).to(DEVICE)
                          for n, h in [(512, 128), (1024, 256), (2048, 512)]]

        with torch.no_grad():
            emb_orig = verifier.encode_batch(waveform)

        for exp_name, cfg in EXPERIMENTS.items():
            outpath = os.path.join(OUTPUT_DIR, f"prot_{base_name}_{exp_name}.wav")
            print(f"   🧪 Training {exp_name}...")

            model = QualityConstrainedAttacker(
                beta_limit=cfg.get("beta_limit", 0.35), sq_limit=cfg.get("sq_limit", 0.50),
                lpc_limit=cfg.get("lpc_limit", 0.02), gain_limit=cfg.get("gain_limit", 0.20)
            ).to(DEVICE)

            optimizer = optim.Adam(model.parameters(), lr=0.008)
            iterations = cfg.get("iterations", 800)
            best_loss, best_state = float('inf'), None

            for step in range(iterations):
                optimizer.zero_grad()
                hacked = model(waveform)

                if torch.isnan(hacked).any():
                    with torch.no_grad(): model.lpc_offsets.data *= 0.3
                    continue

                loss = torch.tensor(0.0, device=DEVICE)

                if "mcd" in cfg: loss += cfg["mcd"] * mcd_attack_loss(waveform, hacked, mfcc_transform)
                if "cosine_embed" in cfg: loss += cfg["cosine_embed"] * cosine_embed_loss(waveform, hacked, verifier, emb_orig)[0]
                if "delta_mfcc" in cfg: loss += cfg["delta_mfcc"] * delta_mfcc_attack_loss(waveform, hacked, mfcc_transform)

                loss += cfg["mel_preserve"] * multi_resolution_mel_loss(waveform, hacked, mel_transforms)
                loss += cfg["waveform_mse"] * waveform_mse_loss(waveform, hacked)
                loss += cfg["snr_floor"] * waveform_snr_loss(waveform, hacked, cfg.get("snr_target", 20.0))
                if "param_reg" in cfg:
                    reg = (torch.sum(model.betas**2) + torch.sum(model.sq_coeffs**2) +
                           torch.sum(model.lpc_offsets**2)*50.0 + torch.sum(model.band_gains**2)*10.0)
                    loss += cfg["param_reg"] * reg

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                optimizer.step()
                model.clamp_params()

                if loss.item() < best_loss and not math.isnan(loss.item()):
                    best_loss = loss.item()
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}

            if best_state: model.load_state_dict(best_state)
            with torch.no_grad():
                final_audio = model(waveform)
                if torch.isnan(final_audio).any(): final_audio = waveform
                torchaudio.save(outpath, final_audio.cpu(), 16000)

            gc.collect(); torch.cuda.empty_cache()

    # ==========================================
    # STAGE B: CLONING ALL FILES
    # ==========================================
    print("\n" + "=" * 70)
    print("🎙️ STAGE B: Cloning with XTTS v2...")
    print("=" * 70)

    from TTS.api import TTS
    CLONE_TEXT = "This is a voice cloning test to evaluate the protection quality of the audio defense system."
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)

    for audio_data in processed_files:
        base_name = audio_data["name"]
        orig_path = audio_data["path"]
        print(f"\n   Cloning variations for: {base_name}")

        bl_path = os.path.join(OUTPUT_DIR, f"clone_{base_name}_BASELINE.wav")
        try:
            tts.tts_to_file(text=CLONE_TEXT, speaker_wav=orig_path, language="en", file_path=bl_path)
        except Exception as e: print(f"  ❌ Baseline Fail: {e}")

        for exp_name in EXPERIMENTS.keys():
            prot_p = os.path.join(OUTPUT_DIR, f"prot_{base_name}_{exp_name}.wav")
            clo_p = os.path.join(OUTPUT_DIR, f"clone_{base_name}_{exp_name}.wav")
            if os.path.exists(prot_p):
                try: tts.tts_to_file(text=CLONE_TEXT, speaker_wav=prot_p, language="en", file_path=clo_p)
                except Exception as e: print(f"  ❌ Clone Fail {exp_name}: {e}")

    del tts; gc.collect(); torch.cuda.empty_cache()

    # ==========================================
    # STAGE C: EVALUATION AND SCOREBOARDS
    # ==========================================
    print("\n" + "=" * 70)
    print("📊 STAGE C: Evaluating & Generating Scoreboards...")
    print("=" * 70)

    all_results = []

    for audio_data in processed_files:
        base_name = audio_data["name"]
        orig_path = audio_data["path"]

        bl_path = os.path.join(OUTPUT_DIR, f"clone_{base_name}_BASELINE.wav")
        if os.path.exists(bl_path):
            cm = calculate_clone_metrics(orig_path, bl_path, verifier)
            all_results.append({"Audio": base_name, "Experiment": "BASELINE", "Clone_CosSim": cm["Clone_CosSim"], "Clone_MCD": cm["Clone_MCD"]})

        for exp_name in EXPERIMENTS.keys():
            row = {"Audio": base_name, "Experiment": exp_name}
            prot_p = os.path.join(OUTPUT_DIR, f"prot_{base_name}_{exp_name}.wav")
            clo_p = os.path.join(OUTPUT_DIR, f"clone_{base_name}_{exp_name}.wav")

            if os.path.exists(prot_p):
                row.update(calculate_protection_metrics(orig_path, prot_p, verifier))
            if os.path.exists(clo_p):
                row.update(calculate_clone_metrics(orig_path, clo_p, verifier))

            cs = safe_float(row.get("CosSim_Prot"), 1.0)
            mcd = safe_float(row.get("MCD_Prot"), 0.0)
            pq = safe_float(row.get("PESQ"), 1.0)
            st = safe_float(row.get("STOI"), 0.0)
            cc = safe_float(row.get("Clone_CosSim"), 1.0)
            snr_v = safe_float(row.get("SNR_dB"), 0.0)

            quality_ok = (pq >= 2.5 and st >= 0.70)
            if quality_ok:
                score = (0.20 * (1.0 - cs) + 0.10 * min(mcd/15.0, 1.0) +
                         0.30 * (1.0 - cc) + 0.20 * st +
                         0.15 * max(0.0, pq/4.5) + 0.05 * min(max(snr_v, 0)/30.0, 1.0))
            else:
                score = 0.1 * (1.0 - cc)

            row["Score"] = round(score, 4)
            row["QualityOK"] = "✅" if quality_ok else "❌"
            all_results.append(row)

    df_ind = pd.DataFrame(all_results)
    ind_csv = os.path.join(OUTPUT_DIR, "individual_audio_scores.csv")
    df_ind.to_csv(ind_csv, index=False)

    df_models = df_ind[df_ind["Experiment"] != "BASELINE"]
    numeric_cols = ["CosSim_Prot", "MCD_Prot", "PESQ", "STOI", "SNR_dB", "Clone_CosSim", "Clone_MCD", "Score"]

    df_mean = df_models.groupby("Experiment")[numeric_cols].mean().reset_index()

    # --------------------------------------------------------
    # THE EER CALCULATION ADDITION (Logic preserved perfectly)
    # --------------------------------------------------------
    print("\n   Calculating Cross-Speaker Baseline for EER...")
    orig_embeds = []
    for audio_data in processed_files:
        wav_o, _ = librosa.load(audio_data["path"], sr=16000)
        t_o = torch.tensor(wav_o).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            orig_embeds.append(verifier.encode_batch(t_o).squeeze())

    # Calculate negative pairs (Impostors)
    neg_scores = []
    for i in range(len(orig_embeds)):
        for j in range(i+1, len(orig_embeds)):
            sim = F.cosine_similarity(orig_embeds[i], orig_embeds[j], dim=0).item()
            neg_scores.append(sim)

    # Calculate EER per experiment
    eer_list = []
    for exp in df_mean["Experiment"]:
        exp_data = df_ind[df_ind["Experiment"] == exp]
        prot_pos = exp_data["CosSim_Prot"].dropna().tolist()
        clone_pos = exp_data["Clone_CosSim"].dropna().tolist()

        prot_eer = compute_eer(prot_pos, neg_scores)
        clone_eer = compute_eer(clone_pos, neg_scores)
        eer_list.append({"Experiment": exp, "Prot_EER_%": prot_eer, "Clone_EER_%": clone_eer})

    df_eer = pd.DataFrame(eer_list)
    df_mean = pd.merge(df_mean, df_eer, on="Experiment")

    # Reorder the columns to place the EER next to Cosine Similarity
    cols = ["Experiment", "CosSim_Prot", "Prot_EER_%", "MCD_Prot", "PESQ", "STOI", "SNR_dB", "Clone_CosSim", "Clone_EER_%", "Clone_MCD", "Score"]
    df_mean = df_mean[cols]

    df_mean = df_mean.sort_values(by="Score", ascending=False).round(4)
    # --------------------------------------------------------

    mean_csv = os.path.join(OUTPUT_DIR, "mean_summary_scoreboard.csv")
    df_mean.to_csv(mean_csv, index=False)

    print(f"\n💾 Saved Individual Scores to: {ind_csv}")
    print(f"💾 Saved Mean Scoreboard to: {mean_csv}\n")

    print("=========================================================================================")
    print(" 🏆 FINAL MEAN SCOREBOARD (Across All 10 LibriSpeech Audios)")
    print("=========================================================================================")
    print(df_mean.to_string(index=False))
    print("=========================================================================================")

run_batch_pipeline()

██████████████████████████████████████████████████████████████████████
█                    PART 1: DOWNLOADING DATASET                     █
██████████████████████████████████████████████████████████████████████
Accessing LibriSpeech (Streaming Mode)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


...

label_encoder.txt: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder



🔧 STAGE A: Training Models Across All Audios...

🎧 Processing Voice: sample_1
   🧪 Training A3_moderate...
   🧪 Training C3_light_preserve...
   🧪 Training B3_mcd_cosine_safe...
   🧪 Training A2_tight...
   🧪 Training A1_ultraTight...

🎧 Processing Voice: sample_2
   🧪 Training A3_moderate...
   🧪 Training C3_light_preserve...
   🧪 Training B3_mcd_cosine_safe...
   🧪 Training A2_tight...
   🧪 Training A1_ultraTight...

🎧 Processing Voice: sample_3
   🧪 Training A3_moderate...
   🧪 Training C3_light_preserve...
   🧪 Training B3_mcd_cosine_safe...
   🧪 Training A2_tight...
   🧪 Training A1_ultraTight...

🎧 Processing Voice: sample_4
   🧪 Training A3_moderate...
   🧪 Training C3_light_preserve...
   🧪 Training B3_mcd_cosine_safe...
   🧪 Training A2_tight...
   🧪 Training A1_ultraTight...

🎧 Processing Voice: sample_5
   🧪 Training A3_moderate...
   🧪 Training C3_light_preserve...
   🧪 Training B3_mcd_cosine_safe...
   🧪 Training A2_tight...
   🧪 Training A1_ultraTight...

🎧 Processing Vo

100%|██████████| 1.87G/1.87G [00:25<00:00, 71.9MiB/s]
4.37kiB [00:00, 6.04MiB/s]
361kiB [00:00, 63.0MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 61.3kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 63.2MiB/s]



   Cloning variations for: sample_1

   Cloning variations for: sample_2

   Cloning variations for: sample_3

   Cloning variations for: sample_4

   Cloning variations for: sample_5

   Cloning variations for: sample_6

   Cloning variations for: sample_7

   Cloning variations for: sample_8

   Cloning variations for: sample_9

   Cloning variations for: sample_10

📊 STAGE C: Evaluating & Generating Scoreboards...

   Calculating Cross-Speaker Baseline for EER...

💾 Saved Individual Scores to: /content/experiments_batch_top5/individual_audio_scores.csv
💾 Saved Mean Scoreboard to: /content/experiments_batch_top5/mean_summary_scoreboard.csv

 🏆 FINAL MEAN SCOREBOARD (Across All 10 LibriSpeech Audios)
        Experiment  CosSim_Prot  Prot_EER_%  MCD_Prot  PESQ   STOI  SNR_dB  Clone_CosSim  Clone_EER_%  Clone_MCD  Score
B3_mcd_cosine_safe       0.9976        0.00    30.647 4.583 0.9986    9.78        0.6252        11.67     33.953 0.5814
          A2_tight       0.9980        0.00    2

In [ ]:
# ============================================================
# HYBRID BATCH PIPELINE v5
# Source-Filter + Waveform Perturbation
# Tested on 10 LibriSpeech speakers
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import torch.optim as optim
import math
import time
import gc
import numpy as np
import pandas as pd
import librosa
from datasets import load_dataset
from speechbrain.inference.speaker import SpeakerRecognition
from pesq import pesq
from pystoi import stoi
import warnings
warnings.filterwarnings("ignore")

os.environ["COQUI_TOS_AGREED"] = "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000

INPUT_DIR = "/content/audio_samples"
OUTPUT_DIR = "/content/experiments_v5"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# PART 1: DOWNLOAD DATASET
# ============================================================

print("█" * 70)
print("█" + " DOWNLOADING DATASET ".center(68) + "█")
print("█" * 70)

AUDIO_FILES = [os.path.join(INPUT_DIR, f"sample_{i}.wav") for i in range(1, 11)]
files_missing = any(not os.path.exists(f) for f in AUDIO_FILES)

if files_missing:
    print("Accessing LibriSpeech (Streaming Mode)...")
    dataset = load_dataset(
        "librispeech_asr", "clean", split="train.100", streaming=True
    ).shuffle(seed=42)
    for i, example in enumerate(dataset.take(10)):
        audio_tensor = torch.tensor(example["audio"]["array"]).unsqueeze(0)
        sampling_rate = example["audio"]["sampling_rate"]
        torchaudio.save(AUDIO_FILES[i], audio_tensor, sampling_rate)
        print(f"   ✅ Saved: {AUDIO_FILES[i]}")
else:
    print("✅ All 10 samples already exist.")
print()


# ============================================================
# MODELS
# ============================================================

class SourceFilterModule(nn.Module):
    """Your source-filter model — completely unchanged."""
    def __init__(self, order=16, n_tilt_stages=4, n_sq_basis=8, n_bands=8,
                 beta_limit=0.35, sq_limit=0.50, lpc_limit=0.02, gain_limit=0.20):
        super().__init__()
        self.order = order
        self.n_tilt_stages = n_tilt_stages
        self.n_sq_basis = n_sq_basis
        self.n_bands = n_bands
        self.beta_limit = beta_limit
        self.sq_limit = sq_limit
        self.lpc_limit = lpc_limit
        self.gain_limit = gain_limit

        self.betas = nn.Parameter(torch.zeros(n_tilt_stages))
        self.sq_coeffs = nn.Parameter(torch.zeros(n_sq_basis))
        self.lpc_offsets = nn.Parameter(torch.zeros(order))
        self.band_gains = nn.Parameter(torch.zeros(n_bands))
        self.register_buffer(
            'band_edges',
            torch.tensor([0, 300, 700, 1200, 2000, 3000, 4500, 6000, 8000],
                         dtype=torch.float32)
        )

    def forward(self, x):
        B, L = x.shape
        dev = x.device

        n_fft = 2 ** math.ceil(math.log2(L * 2 - 1))
        X = torch.fft.rfft(x, n=n_fft)
        autocorr = torch.fft.irfft(X.abs() ** 2, n=n_fft)
        r = autocorr[:, :self.order + 1]
        R = torch.zeros(B, self.order, self.order, device=dev)
        for i in range(self.order):
            for j in range(self.order):
                R[:, i, j] = r[:, abs(i - j)]

        a = torch.linalg.solve(R, r[:, 1:])
        a_orig = torch.cat([torch.ones(B, 1, device=dev), -a], dim=1)
        a_pert = a_orig.clone()
        a_pert[:, 1:] = a_orig[:, 1:] + self.lpc_offsets.unsqueeze(0)

        res = F.conv1d(x.unsqueeze(1),
                       torch.flip(a_orig, [1]).unsqueeze(1),
                       padding=self.order)
        src = res[:, :, :-self.order].squeeze(1)

        E = torch.fft.rfft(src, n=L)
        mag, ph = E.abs(), E.angle()
        fb = torch.linspace(0, math.pi, ph.shape[-1], device=dev)
        disp = torch.zeros_like(fb)
        for k in range(self.n_sq_basis):
            disp = disp + self.sq_coeffs[k] * torch.sin((k + 1) * fb)
        E_w = mag * torch.polar(torch.ones_like(mag), ph + disp)

        A_p = F.pad(a_pert, (0, L - a_pert.shape[1]))
        A_f = torch.fft.rfft(A_p, n=L) + 1e-8
        X_r = E_w / A_f

        nf = X_r.shape[-1]
        fhz = torch.linspace(0, SR / 2, nf, device=dev)
        gp = torch.zeros(nf, device=dev)
        for b in range(self.n_bands):
            lo, hi = self.band_edges[b], self.band_edges[b + 1]
            gp = gp + self.band_gains[b] * (
                torch.sigmoid((fhz - lo) / 50) * torch.sigmoid((hi - fhz) / 50)
            )
        X_r = X_r * torch.exp(gp.unsqueeze(0))
        out = torch.fft.irfft(X_r, n=L)

        for k in range(self.n_tilt_stages):
            tf = torch.stack([
                torch.ones(1, device=dev), -self.betas[k].unsqueeze(0)
            ]).view(1, 1, 2)
            out = F.conv1d(F.pad(out.unsqueeze(1), (1, 0)), tf).squeeze(1)
        return out

    def clamp_params(self):
        with torch.no_grad():
            self.betas.clamp_(-self.beta_limit, self.beta_limit)
            self.sq_coeffs.clamp_(-self.sq_limit, self.sq_limit)
            self.lpc_offsets.clamp_(-self.lpc_limit, self.lpc_limit)
            self.band_gains.clamp_(-self.gain_limit, self.gain_limit)


class HybridAttacker(nn.Module):
    """
    Source-Filter + Waveform Perturbation.
    SF provides structured biological changes.
    Delta provides targeted adversarial perturbation.
    """
    def __init__(self, audio_length, order=16, max_perturb_ratio=0.025,
                 sf_kwargs=None):
        super().__init__()
        sf_kw = sf_kwargs or {}
        self.sf = SourceFilterModule(order=order, **sf_kw)
        self.delta = nn.Parameter(torch.zeros(1, audio_length))
        self.max_ratio = max_perturb_ratio

    def forward(self, x, signal_rms):
        sf_out = self.sf(x)
        max_amp = signal_rms * self.max_ratio
        d = torch.clamp(self.delta, -max_amp, max_amp)
        dl = min(d.shape[-1], sf_out.shape[-1])
        return sf_out[..., :dl] + d[..., :dl]

    def clamp_all(self, signal_rms):
        self.sf.clamp_params()
        ma = (signal_rms * self.max_ratio).item()
        with torch.no_grad():
            self.delta.data.clamp_(-ma, ma)


# ============================================================
# LOSS FUNCTIONS
# ============================================================

def mcd_attack_loss(original, perturbed, mfcc_transform):
    mo = mfcc_transform(original)
    mp = mfcc_transform(perturbed)
    diff = mo[:, 1:14, :] - mp[:, 1:14, :]
    return -torch.mean(torch.sqrt(2.0 * torch.sum(diff ** 2, dim=1) + 1e-8))


def delta_mfcc_attack_loss(original, perturbed, mfcc_transform):
    mo = mfcc_transform(original)[:, 1:14, :]
    mp = mfcc_transform(perturbed)[:, 1:14, :]
    do_ = mo[:, :, 2:] - mo[:, :, :-2]
    dp_ = mp[:, :, 2:] - mp[:, :, :-2]
    ddo = do_[:, :, 2:] - do_[:, :, :-2]
    ddp = dp_[:, :, 2:] - dp_[:, :, :-2]
    return -torch.mean((do_ - dp_) ** 2) - 0.5 * torch.mean((ddo - ddp) ** 2)


def cosine_embed_loss(original, perturbed, verifier, emb_orig):
    emb_pert = verifier.encode_batch(perturbed)
    return F.cosine_similarity(emb_orig.squeeze(), emb_pert.squeeze(), dim=0)


def ltas_divergence_loss(original, perturbed, n_fft=2048):
    window = torch.hann_window(n_fft).to(original.device)
    so = torch.stft(original, n_fft=n_fft, hop_length=512,
                    return_complex=True, window=window)
    sp = torch.stft(perturbed, n_fft=n_fft, hop_length=512,
                    return_complex=True, window=window)
    lo = torch.mean(torch.log(torch.abs(so) + 1e-8), dim=-1)
    lp = torch.mean(torch.log(torch.abs(sp) + 1e-8), dim=-1)
    l2 = -torch.mean((lo - lp) ** 2)
    loc = lo - lo.mean(dim=-1, keepdim=True)
    lpc = lp - lp.mean(dim=-1, keepdim=True)
    corr = torch.sum(loc * lpc, dim=-1) / \
           (torch.norm(loc, dim=-1) * torch.norm(lpc, dim=-1) + 1e-8)
    return l2 + torch.mean(corr)


def multi_resolution_mel_loss(original, perturbed, mel_transforms):
    total = 0.0
    for mt in mel_transforms:
        total += F.l1_loss(torch.log(mt(perturbed) + 1e-7),
                           torch.log(mt(original) + 1e-7))
    return total / len(mel_transforms)


def temporal_envelope_loss(original, perturbed, frame_size=320):
    of = original.unfold(-1, frame_size, frame_size // 2)
    pf = perturbed.unfold(-1, frame_size, frame_size // 2)
    eo = torch.sqrt(torch.mean(of ** 2, dim=-1) + 1e-8)
    ep = torch.sqrt(torch.mean(pf ** 2, dim=-1) + 1e-8)
    return F.mse_loss(ep, eo)


def waveform_snr_loss(original, perturbed, target_snr_db=25.0):
    noise = perturbed - original
    sig = torch.mean(original ** 2)
    noi = torch.mean(noise ** 2) + 1e-10
    return F.relu(target_snr_db - 10 * torch.log10(sig / noi))


def smoothness_loss(delta):
    d2 = delta[:, 2:] - 2 * delta[:, 1:-1] + delta[:, :-2]
    return torch.mean(d2 ** 2)


def spectral_convergence_loss(original, perturbed, n_fft=1024):
    w = torch.hann_window(n_fft).to(original.device)
    mo = torch.abs(torch.stft(original, n_fft=n_fft, hop_length=256,
                               return_complex=True, window=w))
    mp = torch.abs(torch.stft(perturbed, n_fft=n_fft, hop_length=256,
                               return_complex=True, window=w))
    return torch.norm(mo - mp) / (torch.norm(mo) + 1e-8)


# ============================================================
# EER CALCULATION
# ============================================================

def compute_eer(pos_scores, neg_scores):
    if not pos_scores or not neg_scores:
        return 0.0
    scores = np.array(pos_scores + neg_scores)
    labels = np.array([1] * len(pos_scores) + [0] * len(neg_scores))
    desc = np.argsort(scores)[::-1]
    scores, labels = scores[desc], labels[desc]
    fps = np.cumsum(labels == 0)
    tps = np.cumsum(labels == 1)
    far = fps / float(len(neg_scores))
    fnr = 1.0 - (tps / float(len(pos_scores)))
    idx = np.argmin(np.abs(far - fnr))
    return round((far[idx] + fnr[idx]) / 2.0 * 100, 2)


# ============================================================
# EVALUATION HELPERS
# ============================================================

def safe_float(val, default=0.0):
    try:
        return float(val)
    except (ValueError, TypeError):
        return default


def calculate_protection_metrics(orig_path, prot_path, verifier):
    wav_o, _ = librosa.load(orig_path, sr=16000)
    wav_p, _ = librosa.load(prot_path, sr=16000)
    ml = min(len(wav_o), len(wav_p))
    wav_o, wav_p = wav_o[:ml], wav_p[:ml]

    noise = wav_o - wav_p
    snr = 10 * np.log10(np.sum(wav_o ** 2) / (np.sum(noise ** 2) + 1e-8))
    try:
        pesq_val = pesq(16000, np.clip(wav_o, -1, 1), np.clip(wav_p, -1, 1), 'wb')
    except Exception:
        pesq_val = 1.0
    try:
        stoi_val = stoi(wav_o, wav_p, 16000, extended=False)
    except Exception:
        stoi_val = 0.0

    mfcc_o = librosa.feature.mfcc(y=wav_o, sr=16000, n_mfcc=14)[1:]
    mfcc_p = librosa.feature.mfcc(y=wav_p, sr=16000, n_mfcc=14)[1:]
    mt = min(mfcc_o.shape[1], mfcc_p.shape[1])
    mcd_val = float(np.mean(np.sqrt(2.0 * np.sum(
        (mfcc_o[:, :mt] - mfcc_p[:, :mt]) ** 2, axis=0
    ))))

    t_o = torch.tensor(wav_o).unsqueeze(0).to(DEVICE)
    t_p = torch.tensor(wav_p).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        eo = verifier.encode_batch(t_o)
        ep = verifier.encode_batch(t_p)
        cos_sim = F.cosine_similarity(eo.squeeze(), ep.squeeze(), dim=0).item()

    return {
        "CosSim_Prot": round(cos_sim, 4),
        "MCD_Prot": round(mcd_val, 2),
        "PESQ": round(float(pesq_val), 2),
        "STOI": round(float(stoi_val), 4),
        "SNR_dB": round(float(snr), 1),
    }


def calculate_clone_metrics(orig_path, clone_path, verifier):
    wav_o, _ = librosa.load(orig_path, sr=16000)
    wav_c, _ = librosa.load(clone_path, sr=16000)

    t_o = torch.tensor(wav_o).unsqueeze(0).to(DEVICE)
    t_c = torch.tensor(wav_c).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        eo = verifier.encode_batch(t_o)
        ec = verifier.encode_batch(t_c)
        cos_sim = F.cosine_similarity(eo.squeeze(), ec.squeeze(), dim=0).item()

    mfcc_o = librosa.feature.mfcc(y=wav_o, sr=16000, n_mfcc=14)[1:]
    mfcc_c = librosa.feature.mfcc(y=wav_c, sr=16000, n_mfcc=14)[1:]
    clone_mcd = float(np.sqrt(2.0 * np.sum(
        (np.mean(mfcc_o, axis=1) - np.mean(mfcc_c, axis=1)) ** 2
    )))

    return {"Clone_CosSim": round(cos_sim, 4), "Clone_MCD": round(clone_mcd, 2)}


# ============================================================
# UNIFIED TRAINING FUNCTION
# ============================================================

def train_one(waveform, config, verifier, emb_orig, mfcc_t, mel_ts):
    """
    Train a single model on a single audio.
    Returns the protected waveform tensor.
    """
    device = DEVICE
    model_type = config.get("model_type", "sf_only")
    iterations = config.get("iterations", 800)
    lr = config.get("lr", 0.005)
    sig_rms = torch.sqrt(torch.mean(waveform ** 2))

    if model_type == "hybrid":
        sf_kw = {
            k: config[k] for k in
            ["beta_limit", "sq_limit", "lpc_limit", "gain_limit",
             "n_tilt_stages", "n_sq_basis", "n_bands"]
            if k in config
        }
        model = HybridAttacker(
            audio_length=waveform.shape[-1],
            max_perturb_ratio=config.get("max_perturb_ratio", 0.025),
            sf_kwargs=sf_kw,
        ).to(device)
        optimizer = optim.Adam([
            {"params": model.sf.parameters(), "lr": lr},
            {"params": [model.delta], "lr": lr * 3},
        ])
    else:
        model = SourceFilterModule(
            beta_limit=config.get("beta_limit", 0.35),
            sq_limit=config.get("sq_limit", 0.50),
            lpc_limit=config.get("lpc_limit", 0.02),
            gain_limit=config.get("gain_limit", 0.20),
        ).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)

    if config.get("use_scheduler", False):
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, iterations)
    else:
        scheduler = None

    best_loss, best_state = float('inf'), None

    for step in range(iterations):
        optimizer.zero_grad()

        if model_type == "hybrid":
            hacked = model(waveform, sig_rms)
        else:
            hacked = model(waveform)

        if torch.isnan(hacked).any():
            with torch.no_grad():
                if model_type == "hybrid":
                    model.sf.lpc_offsets.data *= 0.3
                    model.delta.data *= 0.5
                else:
                    model.lpc_offsets.data *= 0.3
            continue

        loss = torch.tensor(0.0, device=device)

        # ===== ATTACK =====
        if "mcd" in config:
            loss = loss + config["mcd"] * mcd_attack_loss(
                waveform, hacked, mfcc_t
            )
        if "delta_mfcc" in config:
            loss = loss + config["delta_mfcc"] * delta_mfcc_attack_loss(
                waveform, hacked, mfcc_t
            )
        if "cosine_embed" in config:
            loss = loss + config["cosine_embed"] * cosine_embed_loss(
                waveform, hacked, verifier, emb_orig
            )
        if "ltas" in config:
            loss = loss + config["ltas"] * ltas_divergence_loss(
                waveform, hacked
            )

        # ===== PRESERVATION =====
        if "mel_preserve" in config:
            loss = loss + config["mel_preserve"] * multi_resolution_mel_loss(
                waveform, hacked, mel_ts
            )
        if "temporal_env" in config:
            loss = loss + config["temporal_env"] * temporal_envelope_loss(
                waveform, hacked
            )
        if "snr_floor" in config:
            loss = loss + config["snr_floor"] * waveform_snr_loss(
                waveform, hacked, config.get("snr_target", 25.0)
            )
        if "spectral_conv" in config:
            loss = loss + config["spectral_conv"] * spectral_convergence_loss(
                waveform, hacked
            )

        # Delta-specific
        if model_type == "hybrid":
            if "smoothness" in config:
                loss = loss + config["smoothness"] * smoothness_loss(model.delta)
            if "delta_energy" in config:
                loss = loss + config["delta_energy"] * torch.mean(model.delta ** 2)

        # SF param reg
        sf_mod = model.sf if model_type == "hybrid" else model
        if "param_reg" in config:
            reg = (torch.sum(sf_mod.betas ** 2) +
                   torch.sum(sf_mod.sq_coeffs ** 2) +
                   torch.sum(sf_mod.lpc_offsets ** 2) * 50.0 +
                   torch.sum(sf_mod.band_gains ** 2) * 10.0)
            loss = loss + config["param_reg"] * reg

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        if scheduler:
            scheduler.step()

        if model_type == "hybrid":
            model.clamp_all(sig_rms)
        else:
            model.clamp_params()

        lv = loss.item()
        if lv < best_loss and not math.isnan(lv):
            best_loss = lv
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict(best_state)

    with torch.no_grad():
        if model_type == "hybrid":
            final = model(waveform, sig_rms)
        else:
            final = model(waveform)
        if torch.isnan(final).any():
            final = waveform
        final = torch.clamp(final, -1.0, 1.0)

    return final


# ============================================================
# EXPERIMENT DEFINITIONS
# ============================================================
#
# LESSON FROM YOUR RESULTS:
# Source-filter only with safe clamps = 0% protection.
# We need hybrid models with waveform perturbation.
#
# We also keep 2 SF-only configs as controls to prove
# the hybrid improvement on 10 speakers.

EXPERIMENTS = {

    # ==========================================
    # CONTROLS (SF-only — will show 0% protection)
    # ==========================================

    "CTRL_sf_moderate": {
        "model_type": "sf_only",
        "iterations": 800,
        "beta_limit": 0.35, "lpc_limit": 0.025, "gain_limit": 0.20,
        "mcd": 1.5, "delta_mfcc": 1.0,
        "mel_preserve": 10.0, "snr_floor": 40.0, "snr_target": 15.0,
        "param_reg": 1.0,
    },

    # ==========================================
    # HYBRID: Gentle (PESQ target > 3.5)
    # ==========================================

    "HYB_gentle": {
        "model_type": "hybrid",
        "iterations": 1000,
        "lr": 0.005,
        "max_perturb_ratio": 0.015,
        "beta_limit": 0.30, "lpc_limit": 0.020, "gain_limit": 0.15,
        "mcd": 1.5,
        "cosine_embed": 2.0,
        "mel_preserve": 20.0,
        "temporal_env": 10.0,
        "snr_floor": 60.0,
        "snr_target": 30.0,
        "spectral_conv": 20.0,
        "smoothness": 150.0,
        "delta_energy": 800.0,
        "param_reg": 1.5,
    },

    # ==========================================
    # HYBRID: Moderate (PESQ target > 2.8)
    # ==========================================

    "HYB_moderate": {
        "model_type": "hybrid",
        "iterations": 1200,
        "lr": 0.005,
        "max_perturb_ratio": 0.025,
        "beta_limit": 0.30, "lpc_limit": 0.020, "gain_limit": 0.15,
        "mcd": 2.0,
        "cosine_embed": 2.5,
        "delta_mfcc": 0.8,
        "mel_preserve": 12.0,
        "temporal_env": 8.0,
        "snr_floor": 40.0,
        "snr_target": 25.0,
        "spectral_conv": 12.0,
        "smoothness": 100.0,
        "delta_energy": 400.0,
        "param_reg": 1.0,
    },

    # ==========================================
    # HYBRID: Strong (PESQ target > 2.5)
    # ==========================================

    "HYB_strong": {
        "model_type": "hybrid",
        "iterations": 1200,
        "lr": 0.005,
        "max_perturb_ratio": 0.035,
        "beta_limit": 0.30, "lpc_limit": 0.020, "gain_limit": 0.15,
        "mcd": 2.0,
        "cosine_embed": 3.0,
        "delta_mfcc": 1.0,
        "ltas": 0.5,
        "mel_preserve": 8.0,
        "temporal_env": 5.0,
        "snr_floor": 30.0,
        "snr_target": 20.0,
        "spectral_conv": 8.0,
        "smoothness": 80.0,
        "delta_energy": 200.0,
        "param_reg": 0.5,
    },

    # ==========================================
    # HYBRID: Maximum (push limits)
    # ==========================================

    "HYB_maximum": {
        "model_type": "hybrid",
        "iterations": 1200,
        "lr": 0.005,
        "max_perturb_ratio": 0.045,
        "beta_limit": 0.35, "lpc_limit": 0.025, "gain_limit": 0.20,
        "mcd": 2.5,
        "cosine_embed": 3.0,
        "delta_mfcc": 1.5,
        "ltas": 0.8,
        "mel_preserve": 5.0,
        "snr_floor": 20.0,
        "snr_target": 15.0,
        "spectral_conv": 5.0,
        "smoothness": 50.0,
        "delta_energy": 100.0,
        "param_reg": 0.3,
    },
}


# ============================================================
# MASTER PIPELINE
# ============================================================

def run_batch_pipeline_v5():
    total_start = time.time()
    n_exp = len(EXPERIMENTS)
    n_files = len(AUDIO_FILES)

    print("\n" + "█" * 70)
    print("█" + " v5 HYBRID BATCH PIPELINE ".center(68) + "█")
    print("█" * 70)
    print(f"  Files: {n_files} | Experiments: {n_exp}")
    print(f"  Total runs: {n_files * n_exp}")
    print("█" * 70)

    # Load verifier once
    print("\n📦 Loading ECAPA-TDNN verifier...")
    verifier = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="pretrained_models/spkrec-ecapa-voxceleb",
        run_opts={"device": DEVICE}
    )
    print("✅ Loaded.\n")

    processed_files = []

    # ==========================================
    # STAGE A: TRAIN
    # ==========================================
    print("=" * 70)
    print("🔧 STAGE A: Training all models on all audios...")
    print("=" * 70)

    # Pre-build shared transforms
    mfcc_t = T.MFCC(sample_rate=SR, n_mfcc=40).to(DEVICE)
    mel_ts = [
        T.MelSpectrogram(sample_rate=SR, n_fft=n, hop_length=h, n_mels=80).to(DEVICE)
        for n, h in [(512, 128), (1024, 256), (2048, 512)]
    ]

    for fi, audio_path in enumerate(AUDIO_FILES):
        if not os.path.exists(audio_path):
            continue

        base_name = os.path.basename(audio_path).replace('.wav', '')
        processed_files.append({"path": audio_path, "name": base_name})

        print(f"\n🎧 [{fi+1}/{n_files}] Voice: {base_name}")

        # Load audio once per file
        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000:
            waveform = T.Resample(sr, 16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = waveform.to(DEVICE)

        with torch.no_grad():
            emb_orig = verifier.encode_batch(waveform)

        for ei, (exp_name, cfg) in enumerate(EXPERIMENTS.items()):
            outpath = os.path.join(OUTPUT_DIR, f"prot_{base_name}_{exp_name}.wav")
            mt = cfg.get("model_type", "sf_only")
            print(f"   [{ei+1}/{n_exp}] {exp_name} ({mt})...", end=" ")

            t0 = time.time()
            try:
                protected = train_one(
                    waveform, cfg, verifier, emb_orig, mfcc_t, mel_ts
                )
                torchaudio.save(outpath, protected.cpu(), SR)
                print(f"✅ {time.time()-t0:.0f}s")
            except Exception as e:
                print(f"❌ {e}")

            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    # ==========================================
    # STAGE B: CLONE
    # ==========================================
    print("\n" + "=" * 70)
    print("🎙️ STAGE B: Cloning with XTTS v2...")
    print("=" * 70)

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    from TTS.api import TTS
    CLONE_TEXT = "This is a voice cloning test to evaluate the protection quality of the audio defense system."
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)

    for audio_data in processed_files:
        base_name = audio_data["name"]
        orig_path = audio_data["path"]
        print(f"\n   Cloning {base_name}...")

        # Baseline
        bl_path = os.path.join(OUTPUT_DIR, f"clone_{base_name}_BASELINE.wav")
        try:
            tts.tts_to_file(text=CLONE_TEXT, speaker_wav=orig_path,
                            language="en", file_path=bl_path)
        except Exception as e:
            print(f"     ❌ Baseline: {e}")

        # Protected versions
        for exp_name in EXPERIMENTS:
            prot_p = os.path.join(OUTPUT_DIR, f"prot_{base_name}_{exp_name}.wav")
            clo_p = os.path.join(OUTPUT_DIR, f"clone_{base_name}_{exp_name}.wav")
            if os.path.exists(prot_p):
                try:
                    tts.tts_to_file(text=CLONE_TEXT, speaker_wav=prot_p,
                                    language="en", file_path=clo_p)
                except Exception as e:
                    print(f"     ❌ {exp_name}: {e}")

    del tts
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    # ==========================================
    # STAGE C: EVALUATE
    # ==========================================
    print("\n" + "=" * 70)
    print("📊 STAGE C: Evaluating...")
    print("=" * 70)

    all_results = []

    for audio_data in processed_files:
        base_name = audio_data["name"]
        orig_path = audio_data["path"]

        # Baseline clone
        bl_path = os.path.join(OUTPUT_DIR, f"clone_{base_name}_BASELINE.wav")
        if os.path.exists(bl_path):
            cm = calculate_clone_metrics(orig_path, bl_path, verifier)
            all_results.append({
                "Audio": base_name,
                "Experiment": "BASELINE",
                "Clone_CosSim": cm["Clone_CosSim"],
                "Clone_MCD": cm["Clone_MCD"],
            })

        for exp_name in EXPERIMENTS:
            row = {"Audio": base_name, "Experiment": exp_name}
            prot_p = os.path.join(OUTPUT_DIR, f"prot_{base_name}_{exp_name}.wav")
            clo_p = os.path.join(OUTPUT_DIR, f"clone_{base_name}_{exp_name}.wav")

            if os.path.exists(prot_p):
                row.update(calculate_protection_metrics(orig_path, prot_p, verifier))
            if os.path.exists(clo_p):
                row.update(calculate_clone_metrics(orig_path, clo_p, verifier))

            all_results.append(row)

    df_ind = pd.DataFrame(all_results)

    # ==========================================
    # COMPUTE SCORES
    # ==========================================

    # Get baseline Clone_CosSim per audio for relative scoring
    baseline_clones = df_ind[df_ind["Experiment"] == "BASELINE"].set_index("Audio")["Clone_CosSim"]

    for idx, row in df_ind.iterrows():
        if row["Experiment"] == "BASELINE":
            continue

        cs = safe_float(row.get("CosSim_Prot"), 1.0)
        mcd = safe_float(row.get("MCD_Prot"), 0.0)
        pq = safe_float(row.get("PESQ"), 1.0)
        st = safe_float(row.get("STOI"), 0.0)
        cc = safe_float(row.get("Clone_CosSim"), 1.0)
        snr_v = safe_float(row.get("SNR_dB"), 0.0)

        # Get this audio's baseline for relative comparison
        bl_cc = safe_float(baseline_clones.get(row["Audio"], 0.6), 0.6)

        # Clone improvement: how much did we reduce vs baseline?
        # 0.0 = same as baseline, 1.0 = clone completely broken
        clone_improvement = max(0, (bl_cc - cc) / (bl_cc + 1e-8))

        quality_ok = (pq >= 2.5 and st >= 0.70)

        if quality_ok:
            score = (
                0.10 * (1.0 - cs) +                       # Embedding shift
                0.05 * min(mcd / 15.0, 1.0) +              # Spectral shift
                0.30 * clone_improvement +                  # Clone disruption (main goal)
                0.05 * (1.0 - cc) +                        # Absolute clone distance
                0.20 * st +                                # Intelligibility
                0.15 * max(0.0, pq / 4.5) +                # Quality
                0.10 * min(max(snr_v, 0) / 30.0, 1.0) +   # SNR
                0.05 * min(max(0, 1.0 - cs), 1.0)          # Protection strength
            )
        else:
            # Penalized score for bad quality
            score = 0.15 * clone_improvement

        df_ind.at[idx, "Score"] = round(score, 4)
        df_ind.at[idx, "QualOK"] = "✅" if quality_ok else "❌"
        df_ind.at[idx, "Clone_Improv%"] = round(clone_improvement * 100, 1)

    # Save individual
    ind_csv = os.path.join(OUTPUT_DIR, "individual_scores.csv")
    df_ind.to_csv(ind_csv, index=False)

    # ==========================================
    # MEAN SUMMARY
    # ==========================================

    df_models = df_ind[df_ind["Experiment"] != "BASELINE"].copy()
    numeric_cols = [
        "CosSim_Prot", "MCD_Prot", "PESQ", "STOI", "SNR_dB",
        "Clone_CosSim", "Clone_MCD", "Score", "Clone_Improv%"
    ]
    existing_num = [c for c in numeric_cols if c in df_models.columns]

    for c in existing_num:
        df_models[c] = pd.to_numeric(df_models[c], errors='coerce')

    df_mean = df_models.groupby("Experiment")[existing_num].mean().reset_index()

    # ---- EER CALCULATION ----
    print("\n   Computing EER...")
    orig_embeds = []
    for audio_data in processed_files:
        wav_o, _ = librosa.load(audio_data["path"], sr=16000)
        t_o = torch.tensor(wav_o).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            orig_embeds.append(verifier.encode_batch(t_o).squeeze())

    neg_scores = []
    for i in range(len(orig_embeds)):
        for j in range(i + 1, len(orig_embeds)):
            sim = F.cosine_similarity(orig_embeds[i], orig_embeds[j], dim=0).item()
            neg_scores.append(sim)

    eer_list = []
    for exp in df_mean["Experiment"]:
        exp_data = df_ind[df_ind["Experiment"] == exp]
        prot_pos = exp_data["CosSim_Prot"].dropna().tolist()
        clone_pos = exp_data["Clone_CosSim"].dropna().tolist()
        eer_list.append({
            "Experiment": exp,
            "Prot_EER_%": compute_eer(prot_pos, neg_scores),
            "Clone_EER_%": compute_eer(clone_pos, neg_scores),
        })

    df_eer = pd.DataFrame(eer_list)
    df_mean = pd.merge(df_mean, df_eer, on="Experiment")

    # Baseline mean
    bl_mean = df_ind[df_ind["Experiment"] == "BASELINE"]["Clone_CosSim"].mean()

    # Model type column
    type_map = {name: cfg.get("model_type", "sf_only") for name, cfg in EXPERIMENTS.items()}
    df_mean["Type"] = df_mean["Experiment"].map(type_map)

    # Final column order
    col_order = [
        "Experiment", "Type",
        "CosSim_Prot", "Prot_EER_%",
        "MCD_Prot", "PESQ", "STOI", "SNR_dB",
        "Clone_CosSim", "Clone_EER_%", "Clone_MCD",
        "Clone_Improv%", "Score"
    ]
    cols = [c for c in col_order if c in df_mean.columns]
    df_mean = df_mean[cols].sort_values("Score", ascending=False).round(4)

    mean_csv = os.path.join(OUTPUT_DIR, "mean_scoreboard.csv")
    df_mean.to_csv(mean_csv, index=False)

    # ==========================================
    # REPORT
    # ==========================================
    elapsed = time.time() - total_start

    print(f"\n💾 Saved: {ind_csv}")
    print(f"💾 Saved: {mean_csv}")

    print("\n\n" + "█" * 80)
    print("█" + " FINAL MEAN SCOREBOARD ".center(78) + "█")
    print("█" * 80)
    print(f"\n  Speakers: {n_files} | Experiments: {n_exp}")
    print(f"  Time: {elapsed/60:.1f} minutes")
    print(f"  Baseline Clone_CosSim (mean): {bl_mean:.4f}")

    print("\n  METRIC KEY:")
    print("  Clone_Improv%  = How much BETTER than doing nothing")
    print("                   30% = clone 30% less similar than baseline")
    print("  Clone_EER_%    = Error rate for clone detection")
    print("                   50% = random chance = maximally confused")
    print("  Score          = Composite (higher = better overall)")

    print("\n  RESULTS:")
    print(df_mean.to_string(index=False))

    # Rankings
    print("\n" + "=" * 80)
    print("  🏆 ANALYSIS")
    print("=" * 80)

    # SF vs Hybrid comparison
    sf_rows = df_mean[df_mean["Type"] == "sf_only"]
    hyb_rows = df_mean[df_mean["Type"] == "hybrid"]

    if not sf_rows.empty and not hyb_rows.empty:
        best_sf = sf_rows.iloc[0]
        best_hyb = hyb_rows.sort_values("Score", ascending=False).iloc[0]
        print(f"\n  📊 SOURCE-FILTER vs HYBRID:")
        print(f"  Best SF:     {best_sf['Experiment']}")
        print(f"    Clone={best_sf.get('Clone_CosSim','?')} | "
              f"PESQ={best_sf.get('PESQ','?')} | "
              f"Improv={best_sf.get('Clone_Improv%','?')}%")
        print(f"  Best Hybrid: {best_hyb['Experiment']}")
        print(f"    Clone={best_hyb.get('Clone_CosSim','?')} | "
              f"PESQ={best_hyb.get('PESQ','?')} | "
              f"Improv={best_hyb.get('Clone_Improv%','?')}%")

    # Top overall
    top3 = df_mean.nlargest(3, "Score")
    print(f"\n  🥇 TOP 3:")
    for rank, (_, r) in enumerate(top3.iterrows(), 1):
        medal = ["🥇", "🥈", "🥉"][rank - 1]
        print(f"  {medal} {r['Experiment']}")
        print(f"     Clone_CosSim={r.get('Clone_CosSim','?')} | "
              f"Improv={r.get('Clone_Improv%','?')}% | "
              f"PESQ={r.get('PESQ','?')} | "
              f"EER={r.get('Clone_EER_%','?')}%")

    print(f"\n  Baseline for reference: Clone_CosSim={bl_mean:.4f}")
    print("█" * 80 + "\n")

    return df_mean


# ============================================================
# 🚀 RUN
# ============================================================
results = run_batch_pipeline_v5()

██████████████████████████████████████████████████████████████████████
█                        DOWNLOADING DATASET                         █
██████████████████████████████████████████████████████████████████████
Accessing LibriSpeech (Streaming Mode)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


...

label_encoder.txt: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


✅ Loaded.

🔧 STAGE A: Training all models on all audios...

🎧 [1/10] Voice: sample_1
   [1/5] CTRL_sf_moderate (sf_only)... ✅ 20s
   [2/5] HYB_gentle (hybrid)... ✅ 62s
   [3/5] HYB_moderate (hybrid)... ✅ 78s
   [4/5] HYB_strong (hybrid)... ✅ 89s
   [5/5] HYB_maximum (hybrid)... ✅ 82s

🎧 [2/10] Voice: sample_2
   [1/5] CTRL_sf_moderate (sf_only)... ✅ 19s
   [2/5] HYB_gentle (hybrid)... ✅ 64s
   [3/5] HYB_moderate (hybrid)... ✅ 78s
   [4/5] HYB_strong (hybrid)... ✅ 82s
   [5/5] HYB_maximum (hybrid)... ✅ 83s

🎧 [3/10] Voice: sample_3
   [1/5] CTRL_sf_moderate (sf_only)... ✅ 19s
   [2/5] HYB_gentle (hybrid)... ✅ 63s
   [3/5] HYB_moderate (hybrid)... ✅ 77s
   [4/5] HYB_strong (hybrid)... ✅ 80s
   [5/5] HYB_maximum (hybrid)... ✅ 80s

🎧 [4/10] Voice: sample_4
   [1/5] CTRL_sf_moderate (sf_only)... ✅ 20s
   [2/5] HYB_gentle (hybrid)... ✅ 68s
   [3/5] HYB_moderate (hybrid)... ✅ 83s
   [4/5] HYB_strong (hybrid)... ✅ 86s
   [5/5] HYB_maximum (hybrid)... ✅ 86s

🎧 [5/10] Voice: sample_5
   [1/5] CT

100%|██████████| 1.87G/1.87G [00:23<00:00, 78.8MiB/s]
4.37kiB [00:00, 8.43MiB/s]
361kiB [00:00, 64.1MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 20.2kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 67.6MiB/s]



   Cloning sample_1...

   Cloning sample_2...

   Cloning sample_3...

   Cloning sample_4...

   Cloning sample_5...

   Cloning sample_6...

   Cloning sample_7...

   Cloning sample_8...

   Cloning sample_9...

   Cloning sample_10...

📊 STAGE C: Evaluating...

   Computing EER...

💾 Saved: /content/experiments_v5/individual_scores.csv
💾 Saved: /content/experiments_v5/mean_scoreboard.csv


████████████████████████████████████████████████████████████████████████████████
█                            FINAL MEAN SCOREBOARD                             █
████████████████████████████████████████████████████████████████████████████████

  Speakers: 10 | Experiments: 5
  Time: 59.8 minutes
  Baseline Clone_CosSim (mean): 0.6478

  METRIC KEY:
  Clone_Improv%  = How much BETTER than doing nothing
                   30% = clone 30% less similar than baseline
  Clone_EER_%    = Error rate for clone detection
                   50% = random chance = maximally confused
  Score          = Compo

In [ ]:
import os
from IPython.display import Audio, display, HTML

# We will analyze Speaker 1
base_name = "sample_1"
input_dir = "/content/audio_samples"
output_dir = "/content/experiments_v5"

# The exact experiments from your v5 pipeline
experiments = [
    "CTRL_sf_moderate",
    "HYB_gentle",
    "HYB_moderate",
    "HYB_strong",
    "HYB_maximum"
]

def show_audio(title, file_path):
    if os.path.exists(file_path):
        display(HTML(f"<h4 style='margin-bottom:0px;'>{title}</h4>"))
        display(Audio(file_path))
    else:
        display(HTML(f"<h4 style='margin-bottom:0px;'>{title}</h4><p style='color:red;'>File not found.</p>"))

display(HTML(f"<h2>🎧 AUDIO PLAYBACK: {base_name.upper()}</h2><hr>"))

# 1. The Ground Truth
show_audio("1. Original Human Voice", os.path.join(input_dir, f"{base_name}.wav"))

# 2. The Unprotected Clone
show_audio("2. Baseline AI Clone (No Protection)", os.path.join(output_dir, f"clone_{base_name}_BASELINE.wav"))

# 3. The Protections vs. The AI
for exp in experiments:
    display(HTML("<br><hr style='border:1px dashed gray'>"))

    # How does the defense sound to humans?
    prot_path = os.path.join(output_dir, f"prot_{base_name}_{exp}.wav")
    show_audio(f"🛡️ Protected Voice ({exp})", prot_path)

    # Did XTTS break and output garbage?
    clone_path = os.path.join(output_dir, f"clone_{base_name}_{exp}.wav")
    show_audio(f"🤖 Attempted XTTS Clone ({exp})", clone_path)

In [ ]:
# Compress the original samples and all v5 experiment results
!zip -r /content/V5_Adversarial_Results.zip /content/experiments_v5 /content/audio_samples

# Trigger the automatic download to your local machine
from google.colab import files
files.download('/content/V5_Adversarial_Results.zip')

  adding: content/experiments_v5/ (stored 0%)
  adding: content/experiments_v5/clone_sample_7_HYB_gentle.wav (deflated 20%)
  adding: content/experiments_v5/clone_sample_3_HYB_maximum.wav (deflated 20%)
  adding: content/experiments_v5/clone_sample_2_HYB_maximum.wav (deflated 24%)
  adding: content/experiments_v5/prot_sample_4_HYB_gentle.wav (deflated 14%)
  adding: content/experiments_v5/prot_sample_1_HYB_gentle.wav (deflated 16%)
  adding: content/experiments_v5/prot_sample_1_CTRL_sf_moderate.wav (deflated 17%)
  adding: content/experiments_v5/prot_sample_1_HYB_strong.wav (deflated 20%)
  adding: content/experiments_v5/prot_sample_7_HYB_strong.wav (deflated 20%)
  adding: content/experiments_v5/clone_sample_10_HYB_maximum.wav (deflated 23%)
  adding: content/experiments_v5/clone_sample_8_HYB_gentle.wav (deflated 17%)
  adding: content/experiments_v5/clone_sample_7_HYB_maximum.wav (deflated 19%)
  adding: content/experiments_v5/prot_sample_8_CTRL_sf_moderate.wav (deflated 19%)
  addin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>